# M and A, workshop 3 — Loading to SQL, and proving it worked

You have three cleaned files. The last step of the migration is putting them in the database —
and then the step nobody teaches: **showing that what came out is what went in.**

A migration that loses 4 % of its rows looks exactly like one that loses none, right up until a
colleague asks why the customer count dropped. Today you build the checks that answer that
question before anyone asks it.

1. **Build the merged table** from the three real exports.
2. **CSV to SQLite**, with the types you meant.
3. **Reconciliation**: count, nulls, keys, round-trip.
4. **Break it on purpose**, and watch a check catch it.

In [ ]:
# ABOUTME: M and A workshop 3 - loading the merged Only Wood customers into SQLite.
# ABOUTME: Explicit schema, row/null/key reconciliation, and a round-trip diff.
import csv
import io
import sqlite3
from pathlib import Path

import pandas as pd

from pathlib import Path

def qwasar_file(name):
    """Locate a dataset fetched by workshops/get_data.py, from wherever this notebook was opened."""
    for up in range(4):
        p = Path(*([".."] * up), "data", name)
        if p.exists():
            return p
    raise FileNotFoundError(
        f"{name} is missing. Run `uv run get_data.py` in the workshops/ folder first - "
        "these notebooks only ever use the real datasets Qwasar provides.")

TABLE_1 = qwasar_file("only_wood_customer_us_1.csv")
TABLE_2 = qwasar_file("only_wood_customer_us_2.csv")
TABLE_3 = qwasar_file("only_wood_customer_us_3.csv")

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 20)
print("the three exports Only Wood handed over:")
for p in (TABLE_1, TABLE_2, TABLE_3):
    print(f"  {p.name}: {p.stat().st_size / 1000:.0f} kB, {sum(1 for _ in p.open()):,} lines")

DB = Path("customers.db")
if DB.exists():
    DB.unlink()          # a workshop should start from nothing every time it is run

the three exports Only Wood handed over:
  only_wood_customer_us_1.csv: 748 kB, 10,001 lines
  only_wood_customer_us_2.csv: 620 kB, 10,000 lines
  only_wood_customer_us_3.csv: 1101 kB, 10,001 lines


## 1. The merged table

The normalisers from workshop 2, applied to all three files. Read it once — this is the pipeline
your `my_m_and_a` function performs — then we move on to the part this workshop is about.

In [ ]:
SCHEMA_COLUMNS = ["gender", "firstname", "lastname", "email", "age", "city", "country",
                  "created_at", "referral"]

strip_types = lambda s: (s.astype(str).str.replace(r'^"|"$', "", regex=True)
                          .str.replace(r"^(string|integer|character|boolean|float)_", "", regex=True))
clean_text = lambda s: (s.astype(str).str.replace(r'[\\"]', "", regex=True)
                         .str.strip().replace({"": None, "nan": None}))

def normalise_gender(s):
    v = clean_text(strip_types(s)).str.lower()
    return v.map({"male": "Male", "m": "Male", "0": "Male",
                  "female": "Female", "f": "Female", "1": "Female"})

def normalise_country(s):
    key = clean_text(s).str.lower().str.replace(r"[^a-z]", "", regex=True)
    usa = {"us", "usa", "unitedstateofamerica", "unitedstatesofamerica", "unitedstates", "america"}
    return key.map(lambda k: "United States of America" if k in usa else None)

def normalise_city(s):
    return (clean_text(s).str.replace(r"[_\-]+", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.title())

def valid_email(s):
    v = clean_text(strip_types(s)).str.lower()
    return v.where(v.str.match(r"^[^@\s]+@[^@\s]+\.[a-z]{2,}$", na=False))

def normalise_age(s):
    v = pd.to_numeric(clean_text(strip_types(s)).str.extract(r"(\d+)", expand=False), errors="coerce")
    return v.where(v.between(0, 120)).astype("Int64")

def split_name(full):
    parts = clean_text(full).fillna("").str.split(r"\s+", n=1, expand=True)
    if parts.shape[1] == 1:
        parts[1] = None
    return parts[0].str.title().replace({"": None}), parts[1].str.title().replace({"": None})

t1 = pd.read_csv(TABLE_1)
t2 = pd.read_csv(TABLE_2, sep=";", header=None, names=["age", "city", "gender", "name", "email"])
_h, _b = TABLE_3.read_text().split("\n", 1)
t3 = pd.read_csv(io.StringIO(_b), sep="\t", header=None, names=_h.strip().split(","), quoting=csv.QUOTE_NONE)

f2, l2 = split_name(t2.name)
f3, l3 = split_name(strip_types(t3.Name))
frames = [
    pd.DataFrame({"gender": normalise_gender(t1.Gender), "firstname": clean_text(t1.FirstName).str.title(),
                  "lastname": clean_text(t1.LastName).str.title(), "email": valid_email(t1.Email),
                  "age": normalise_age(t1.Age), "city": normalise_city(t1.City),
                  "country": normalise_country(t1.Country)}),
    pd.DataFrame({"gender": normalise_gender(t2.gender), "firstname": f2, "lastname": l2,
                  "email": valid_email(t2.email), "age": normalise_age(t2.age),
                  "city": normalise_city(t2.city), "country": None}),
    pd.DataFrame({"gender": normalise_gender(strip_types(t3.Gender)), "firstname": f3, "lastname": l3,
                  "email": valid_email(strip_types(t3.Email)), "age": normalise_age(t3.Age),
                  "city": normalise_city(strip_types(t3.City)),
                  "country": normalise_country(strip_types(t3.Country))}),
]
merged = pd.concat(frames, ignore_index=True)
merged["created_at"] = None      # no source system recorded it
merged["referral"] = None
merged = (merged.dropna(subset=["email"])
          .sort_values(["email", "country"], na_position="last")
          .drop_duplicates(subset="email", keep="first")
          .reset_index(drop=True))[SCHEMA_COLUMNS]

print(f"30,000 rows in -> {len(merged):,} customers out")
merged.head()

30,000 rows in -> 28,486 customers out


,gender,firstname,lastname,email,age,city,country,created_at,referral
0,Male,Aaron,Jacobi,aaron.jacobi@yahoo.com,81,Fort Worth,None,None,None
1,Male,Aaron,Wisozk,aaron.wisozk@yahoo.com,35,San Antonio,None,None,None
2,Male,Aaron,Lindgren,aaron_lindgren@hotmail.com,49,New York,None,None,None
3,Female,Abbey,Schaefer,abbey_schaefer@yahoo.com,47,Philadelphia,None,None,None
4,Female,Abbie,Barton,abbie.barton@yahoo.com,98,New York,None,None,None


**28 486 customers**, keyed on email, from 30 000 rows. Note what a realistic result looks like:
`created_at` and `referral` are empty for everyone because no source had them, and plenty of
countries are missing because table 2 never recorded one.

## 2. Getting it into SQLite

`sqlite3` ships with Python; there is no server to install. pandas can do the whole load with
`DataFrame.to_sql`, and that is what you would use at work. Your Qwasar `csv_to_sql` does it by
hand with `INSERT` statements, which is worth doing once because it forces you to see the column
list, the placeholders and the commit.

**Rule of thumb:** always `commit()`. A SQLite write that is never committed is a write that never
happened, and it fails silently.

In [ ]:
with sqlite3.connect(DB) as conn:
    merged.to_sql("customers", conn, index=False, if_exists="replace")
    print(pd.read_sql("SELECT COUNT(*) AS rows FROM customers", conn).to_string(index=False))
    print()
    print(pd.read_sql("PRAGMA table_info(customers)", conn)[["name", "type"]].to_string(index=False))

 rows
28486

      name    type
    gender    TEXT
 firstname    TEXT
  lastname    TEXT
     email    TEXT
       age INTEGER
      city    TEXT
   country    TEXT
created_at    TEXT
  referral    TEXT


Look at the `type` column. pandas inferred `TEXT` for the strings and — because the ages went
through a nullable integer — it may well have stored your age column as something you did not
intend. Nobody is 37.0 years old, and a date stored as text will not sort as a date.

This is the first thing a reconciliation check would catch, and it is a one-line fix at load time.

### Exercise 1 — load with the types you meant

Write `load_customers(df, db_path)`: create the table with an **explicit schema** (ages as
`INTEGER`, everything else `TEXT`, email `NOT NULL`), then insert the rows.

The `CREATE TABLE` is given; the exercise is the insert. A SQLite insert takes one `?` per column
and a sequence of row tuples.

In [ ]:
SCHEMA = """
CREATE TABLE customers (
    gender     TEXT,
    firstname  TEXT,
    lastname   TEXT,
    email      TEXT NOT NULL,
    age        INTEGER,
    city       TEXT,
    country    TEXT,
    created_at TEXT,
    referral   TEXT
)
"""

def load_customers(df, db_path):
    """Create the customers table with an explicit schema and insert every row."""
    if Path(db_path).exists():
        Path(db_path).unlink()
    placeholders = ",".join("?" * df.shape[1])
    rows = list(df.astype(object).where(df.notna(), None).itertuples(index=False, name=None))
    with sqlite3.connect(db_path) as conn:
        conn.execute(SCHEMA)
        conn.executemany(f"INSERT INTO customers VALUES ({placeholders})", rows)
        conn.commit()
    return db_path

load_customers(merged, "customers_typed.db")
with sqlite3.connect("customers_typed.db") as conn:
    print(pd.read_sql("PRAGMA table_info(customers)", conn)[["name", "type"]].to_string(index=False))

      name    type
    gender    TEXT
 firstname    TEXT
  lastname    TEXT
     email    TEXT
       age INTEGER
      city    TEXT
   country    TEXT
created_at    TEXT
  referral    TEXT


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

def _query(sql, db="customers_typed.db"):
    with sqlite3.connect(db) as conn:
        return pd.read_sql(sql, conn)

check("the table exists and has rows", lambda: len(_query("SELECT * FROM customers LIMIT 5")) > 0)
check("every customer arrived", lambda: _query("SELECT COUNT(*) n FROM customers").n.item() == len(merged))
check("age is an INTEGER column",
      lambda: _query("PRAGMA table_info(customers)").set_index("name").type.get("age") == "INTEGER")
check("missing ages stayed missing",
      lambda: _query("SELECT COUNT(*) n FROM customers WHERE age IS NULL").n.item()
              == int(merged.age.isna().sum()))

  the table exists and has rows: PASS
  every customer arrived: PASS
  age is an INTEGER column: PASS
  missing ages stayed missing: PASS


True

## 3. Reconciliation: four questions

The load reported no error. That is not evidence. Ask the database what it holds and compare
against what you sent:

| check | what it catches |
|---|---|
| **row count** | rows silently dropped by a failed insert or a filter |
| **null rate per column** | a column that arrived empty because of a name mismatch |
| **key uniqueness** | duplicates created by loading twice |
| **round trip** | values mangled in transit (types, encodings, truncation) |

### Exercise 2 — write the checks

In [ ]:
def reconcile(df, db_path):
    """Compare the source DataFrame against what the database actually stored."""
    with sqlite3.connect(db_path) as conn:
        loaded = pd.read_sql("SELECT * FROM customers", conn)

    src_nulls, out_nulls = int(df.isna().sum().sum()), int(loaded.isna().sum().sum())
    checks = [
        {"check": "row count", "source": len(df), "loaded": len(loaded),
         "passed": len(df) == len(loaded)},
        {"check": "distinct emails", "source": int(df.email.nunique()),
         "loaded": int(loaded.email.nunique()), "passed": df.email.nunique() == loaded.email.nunique()},
        {"check": "total nulls", "source": src_nulls, "loaded": out_nulls,
         "passed": src_nulls == out_nulls},
        {"check": "column order", "source": len(df.columns), "loaded": len(loaded.columns),
         "passed": list(df.columns) == list(loaded.columns)},
    ]
    return pd.DataFrame(checks)

reconcile(merged, "customers_typed.db")

,check,source,loaded,passed
0,row count,28486,28486,True
1,distinct emails,28486,28486,True
2,total nulls,71169,71169,True
3,column order,9,9,True


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

_rec = reconcile(merged, "customers_typed.db")
check("at least three checks", lambda: len(_rec) >= 3)
check("there is a passed column", lambda: "passed" in _rec.columns)
check("a correct load passes everything", lambda: bool(_rec.passed.all()))

  at least three checks: PASS
  there is a passed column: PASS
  a correct load passes everything: PASS


True

### Exercise 3 — break it, and catch it

A check you have never seen fail is a check you do not know works. Load the table a second time
**without clearing it** — the classic mistake of re-running a migration script — and confirm your
reconciliation notices.

In [ ]:
placeholders = ",".join("?" * merged.shape[1])
rows = list(merged.astype(object).where(merged.notna(), None).itertuples(index=False, name=None))
with sqlite3.connect("customers_typed.db") as conn:
    conn.executemany(f"INSERT INTO customers VALUES ({placeholders})", rows)
    conn.commit()

broken = reconcile(merged, "customers_typed.db")
broken

,check,source,loaded,passed
0,row count,28486,56972,False
1,distinct emails,28486,28486,True
2,total nulls,71169,142338,False
3,column order,9,9,True


Two checks go red and they tell different stories: the **row count doubled**, but the
**distinct-email count did not move**. That pair is the signature of a duplicate load, not of extra
customers — and it is exactly why counting rows alone is not enough.

**Rule of thumb:** make your load repeatable. Dropping and recreating the table, or an
`INSERT ... ON CONFLICT` on a unique email, means running the script twice is harmless.

### Exercise 4 — the round trip

The strongest check: pull everything back out and compare it to what you sent, value by value.
If `csv_to_sql` and `sql_to_csv` are inverses, this is empty.

In [ ]:
load_customers(merged, "customers_typed.db")      # start from a clean, correct load again

def round_trip_diff(df, db_path):
    """Return the rows that differ between the source frame and the database contents."""
    with sqlite3.connect(db_path) as conn:
        loaded = pd.read_sql("SELECT * FROM customers", conn)

    def canonical(frame):
        # Compare as text: SQLite returns 85 where pandas held 85.0, and that is not a difference.
        out = frame[list(df.columns)].astype(object).where(frame.notna(), "")
        for col in out.columns:
            out[col] = out[col].map(lambda v: str(int(v)) if isinstance(v, float) and v.is_integer() else str(v))
        return out.sort_values(list(df.columns)).reset_index(drop=True)

    a, b = canonical(df), canonical(loaded)
    unequal = (a != b).any(axis=1)
    return a[unequal].join(b[unequal], rsuffix="_db")

diff = round_trip_diff(merged, "customers_typed.db")
print(f"{len(diff)} differing rows")
diff.head()

0 differing rows


,gender,firstname,lastname,email,age,city,country,created_at,referral,gender_db,firstname_db,lastname_db,email_db,age_db,city_db,country_db,created_at_db,referral_db


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("the round trip is clean", lambda: len(round_trip_diff(merged, "customers_typed.db")) == 0)

  the round trip is clean: PASS


True

Zero differing rows across 28 486 customers. *Now* you can tell your manager the migration is
done, and show them why.

## 4. One more thing: do not commit the data

The project brief is blunt — if the acquired company's CSVs are found in your repository at the
peer review, the project fails. The same rule applies at work for a less dramatic reason: customer
data does not belong in version control, where it is copied to every laptop that clones the repo
and lives forever in the history.

```
# .gitignore
data/
*.db
*.csv
```

Your code must therefore **receive** its input rather than carry it. That is also why
`my_m_and_a` takes CSV *contents* as arguments instead of filenames.

## What to take away

1. **Declare your schema.** Letting the loader guess gives you ages of 37.0 and dates stored as text.
2. **A load without an error is not a verified load.** Count rows, count nulls, count keys.
3. **Row count alone is not enough** — rows doubled while distinct emails stayed flat is a
   duplicate load, and only the pair of checks tells you that.
4. **Make loads repeatable**, so running the script twice is not an incident.
5. **Round-trip when you can.** Out and back, compared as text, is the closest thing to proof.
6. **Never commit the customer data.**

### Take it home
Run your own `my_m_and_a` output through `csv_to_sql`, then write the four checks above against it.
Bring the reconciliation table next session — and if a check fails, bring that too, it is the more
interesting case.